# Evidence Layer v4 — Sanity Check & Comparaison v3 vs v4

Ce notebook vérifie que `EvidenceLayerV4` :
1. Produit les bonnes formes tensorielles
2. Se comporte comme une identité à l'initialisation (gate ≈ 1 → h_ev ≈ dec_output)
3. Mesure la contribution effective de M via `gate_deviation()`
4. Calcule correctement la régularisation ElasticNet
5. Se compare proprement à v3 (mélange additif)

**Dimensions cibles (identiques à B5 sur Cluster 4) :**

| Symbole | Valeur | Signification |
|---------|--------|---------------|
| B | 4 | batch size |
| P | 120 | pred_len (2 heures en minutes) |
| C | 240 | context_length |
| D | 32 | d_model |

**Référence architecturale :**
- v3 : `h_ev = (1-mix)*dec_output + mix*LN(bmm(M, enc_hidden))`
- v4 : `gate = 1 + tanh(Linear(LN(bmm(M, enc_hidden))))`,  `h_ev = dec_output * gate`

## 1 — Setup

In [ ]:
import os, sys
from pathlib import Path

# ── Détection automatique de l'environnement ─────────────────────────────────
try:
    from google.colab import drive
    REPO_URL = 'https://github.com/theblackmamba08/recherche-m2-xai-faas.git'
    REPO_DIR = '/content/recherche-m2-xai-faas'
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        get_ipython().system(f'git -C {REPO_DIR} pull')
    CODE_DIR = f'{REPO_DIR}/code'
    IN_COLAB = True
    print('Environnement : Google Colab')
except ImportError:
    # Local — notebook est dans code/notebooks/, code src dans code/
    CODE_DIR = str(Path('..').resolve())
    IN_COLAB = False
    print('Environnement : local')

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'CODE_DIR = {CODE_DIR}')
print('src/models :', os.listdir(os.path.join(CODE_DIR, 'src', 'models')))

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print(f'PyTorch : {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')

# Seed de reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
from src.models.evidence_layer_v4 import EvidenceLayerV4, evidence_regularization_loss

print('Import EvidenceLayerV4 : OK')

## 2 — Dimensions FAYAM (Cluster 4, Run B5)

In [ ]:
# ── Hyperparams identiques à B5 / PLAN-MEMOIRE.md ────────────────────────────
B = 4     # batch size
P = 120   # pred_len  (2 heures × 60 min)
C = 240   # context_length
D = 32    # d_model

print(f'B={B}  P={P}  C={C}  D={D}')
print()
print('Tenseurs simulés :')

# Simule dec_output  [B, P, D] : sortie du décodeur Transformer
dec_output = torch.randn(B, P, D, device=device)
# Simule enc_hidden  [B, C, D] : états cachés de l'encodeur
enc_hidden = torch.randn(B, C, D, device=device)

print(f'  dec_output : {tuple(dec_output.shape)}')
print(f'  enc_hidden : {tuple(enc_hidden.shape)}')

## 3 — Instanciation + forward pass

In [ ]:
layer_v4 = EvidenceLayerV4(d_model=D, context_length=C).to(device)
print('Paramètres entraînables :')
total = 0
for name, p in layer_v4.named_parameters():
    print(f'  {name:30s}  {tuple(p.shape)}  ({p.numel()} params)')
    total += p.numel()
print(f'  Total : {total} paramètres')

In [ ]:
h_ev, M = layer_v4(dec_output, enc_hidden)

print('Shapes en sortie :')
print(f'  h_ev : {tuple(h_ev.shape)}   attendu : ({B}, {P}, {D})')
print(f'  M    : {tuple(M.shape)}   attendu : ({B}, {P}, {C})')

# ── Assertions dures ──────────────────────────────────────────────────────────
assert h_ev.shape == (B, P, D),  f'Shape h_ev incorrecte : {h_ev.shape}'
assert M.shape    == (B, P, C),  f'Shape M incorrecte    : {M.shape}'
assert (M >= 0).all() and (M <= 1).all(), 'M contient des valeurs hors [0,1]'
assert torch.allclose(M.sum(dim=-1), torch.ones(B, P, device=device), atol=1e-5), \
    'M ne somme pas à 1 sur dim=-1 (softmax cassé)'

print()
print('Propriétés de M (Evidence Map) :')
print(f'  M.min()  = {M.min().item():.6f}   (attendu >= 0)')
print(f'  M.max()  = {M.max().item():.6f}   (attendu <= 1)')
print(f'  M.sum(dim=-1) → mean = {M.sum(dim=-1).mean().item():.6f}  (attendu = 1.0)')

print()
print('Assertions : OK ✅')

## 4 — Vérification identité à l'initialisation

Avec `gate_proj.weight ~ N(0, 0.01)` et `gate_proj.bias = 0` :
- `Linear(h_context) ≈ 0`
- `gate = 1 + tanh(0) = 1`
- `h_ev = dec_output × 1 = dec_output`

Ce comportement préserve les poids du checkpoint B5 lors du fine-tuning.

In [ ]:
# Réinitialisation contrôlée : gate_proj exactement à zéro (init parfaite)
layer_v4_zero = EvidenceLayerV4(d_model=D, context_length=C).to(device)
nn.init.zeros_(layer_v4_zero.gate_proj.weight)
nn.init.zeros_(layer_v4_zero.gate_proj.bias)

with torch.no_grad():
    h_ev_zero, M_zero = layer_v4_zero(dec_output, enc_hidden)

    # Vérification gate interne manuellement
    h_context = layer_v4_zero.layer_norm(torch.bmm(M_zero, enc_hidden))
    gate_val   = 1.0 + torch.tanh(layer_v4_zero.gate_proj(h_context))

print('Test identité (gate_proj = 0) :')
print(f'  gate  : min={gate_val.min().item():.6f}  max={gate_val.max().item():.6f}  '
      f'mean={gate_val.mean().item():.6f}   (attendu = 1.0)')

diff = (h_ev_zero - dec_output).abs()
print(f'  |h_ev - dec_output| : max={diff.max().item():.2e}  mean={diff.mean().item():.2e}')
print(f'  Identité parfaite ? {"OUI ✅" if diff.max().item() < 1e-5 else "NON ❌"}')

print()
print('Avec gate_proj ~ N(0, 0.01) (init réelle de v4) :')
dev = layer_v4.gate_deviation(dec_output, enc_hidden)
print(f'  gate_deviation = {dev:.4f}   (≈ 0 à l\'init, > 0 après entraînement)')

## 5 — Régularisation ElasticNet sur M

$$L_{\text{total}} = L_{\text{forecast}} + \alpha\|M\|_1 + \beta\|M\|_2^2 + \gamma H(M)$$

- **L1** (α) : sparsité — force M vers peu d'entrées actives
- **L2** (β) : stabilité — punit les entrées très grandes
- **H(M)** (γ) : anti-collapse — empêche M d'être trop uniforme (entropie maximale)

In [ ]:
# Paramètres identiques à Run B5
ALPHA = 0.0    # pas de L1 dans B5
BETA  = 1e-3   # L2 activé
GAMMA = 1e-3   # entropie activée

reg_loss = evidence_regularization_loss(M, alpha=ALPHA, beta=BETA, gamma=GAMMA)

# Composantes détaillées
l1_val = M.abs().mean().item()
l2_val = M.pow(2).mean().item()
H_val  = -(M * torch.log(M + 1e-8)).sum(dim=-1).mean().item()

print('Régularisation ElasticNet :')
print(f'  L1  = {l1_val:.6f}  (× α={ALPHA})  → {ALPHA * l1_val:.6f}')
print(f'  L2  = {l2_val:.6f}  (× β={BETA})   → {BETA * l2_val:.6f}')
print(f'  H   = {H_val:.4f}   (× γ={GAMMA})  → {GAMMA * H_val:.6f}')
print(f'  L_reg = {reg_loss.item():.6f}')
print()
print('Interprétation :')
print(f'  H = log(C) = {np.log(C):.3f}  ← entropie max (M uniforme)')
print(f'  H observée = {H_val:.3f}  ← {H_val/np.log(C):.1%} de l\'entropie max')

## 6 — Comparaison v3 (additif) vs v4 (gating)

On simule un forward v3 à la main (sans charger le checkpoint) pour
comparer directement les deux mécanismes sur les mêmes entrées.

In [ ]:
def evidence_layer_v3_forward(dec_output, enc_hidden, mix=0.05):
    """Réplique la logique v3 (mix additif + LayerNorm)."""
    B, P, D = dec_output.shape
    _, C, _  = enc_hidden.shape

    # evidence_proj : [B,P,D] → [B,P,C]
    evidence_proj = nn.Linear(D, C, bias=True)
    evidence_norm = nn.LayerNorm(D)
    nn.init.normal_(evidence_proj.weight, std=0.01)
    nn.init.zeros_(evidence_proj.bias)

    with torch.no_grad():
        M = torch.softmax(evidence_proj(dec_output), dim=-1)
        h_evidence = torch.bmm(M, enc_hidden)
        h_evidence = evidence_norm(h_evidence)
        h_ev = (1.0 - mix) * dec_output + mix * h_evidence
    return h_ev, M


mix = 0.25  # valeur d'inférence B5
h_ev_v3, M_v3 = evidence_layer_v3_forward(dec_output, enc_hidden, mix=mix)
h_ev_v4, M_v4 = layer_v4(dec_output, enc_hidden)

print('Comparaison v3 vs v4 sur tenseurs aléatoires :')
print()
print('                         v3 (additif, mix=0.25)   v4 (gate, 1+tanh)')
print(f'  h_ev.mean()          : {h_ev_v3.mean().item():>22.4f}   {h_ev_v4.mean().item():>18.4f}')
print(f'  h_ev.std()           : {h_ev_v3.std().item():>22.4f}   {h_ev_v4.std().item():>18.4f}')
print(f'  |h_ev - dec_output|  : {(h_ev_v3 - dec_output).abs().mean().item():>22.4f}   {(h_ev_v4 - dec_output).abs().mean().item():>18.4f}')
print()
print('Propriétés garanties :')
print(f'  v3 : h_ev = (1-mix)*dec + mix*LN(bmm)   →  déviation toujours = mix × LN(bmm)')
print(f'  v4 : h_ev = dec * gate,  gate ∈ (0,2)    →  déviation adaptive selon M')

## 7 — Visualisation : gate, M, et comparaison architecturale

In [ ]:
with torch.no_grad():
    h_context = layer_v4.layer_norm(torch.bmm(M_v4, enc_hidden))
    gate = 1.0 + torch.tanh(layer_v4.gate_proj(h_context))

# Sélection de l'exemple 0 du batch pour visualisation
M_np    = M_v4[0].cpu().numpy()      # (P, C)
gate_np = gate[0].cpu().numpy()      # (P, D)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ── (a) Carte M ──────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(M_np, aspect='auto', cmap='YlOrRd', origin='upper')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
ax.set_title('(a) Evidence Map M\n[P=120, C=240]')
ax.set_xlabel('Position encodeur (contexte)')
ax.set_ylabel('Horizon t (prédiction)')

# ── (b) Gate ─────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(gate_np, aspect='auto', cmap='RdBu_r', vmin=0, vmax=2, origin='upper')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
ax.set_title('(b) Gate = 1 + tanh(Linear(h_ctx))\n[P=120, D=32]  — centré sur 1')
ax.set_xlabel('Dimension D du modèle')
ax.set_ylabel('Horizon t (prédiction)')

# ── (c) Distribution du gate ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
ax.hist(gate_np.flatten(), bins=50, color='#BE185D', edgecolor='white', alpha=0.85)
ax.axvline(1.0, color='black', ls='--', lw=2, label='identité (gate=1)')
ax.axvline(gate_np.mean(), color='crimson', ls='-', lw=1.5,
           label=f'mean={gate_np.mean():.3f}')
ax.set_xlabel('Valeur du gate')
ax.set_ylabel('Fréquence')
ax.set_title('(c) Distribution gate\n(init: std=0.01 → concentrée sur 1)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── (d) Argmax de M par horizon ──────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
argmax_M = M_np.argmax(axis=1)
ax.plot(range(P), argmax_M, color='#166534', lw=1.5)
ax.set_xlabel('Horizon t')
ax.set_ylabel('Position encodeur argmax(M[t,:])')
ax.set_title('(d) argmax(M) par horizon\n(diagonale = alignement temporel)')
ax.set_ylim(0, C)
# Ligne diagonale (alignement parfait)
diag_factor = C / P
ax.plot(range(P), [t * diag_factor for t in range(P)],
        ls='--', color='gray', alpha=0.5, label='diagonale parfaite')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── (e) Effet du gate sur dec_output (feature 0) ──────────────────────────────
ax = fig.add_subplot(gs[1, 1])
dec_np = dec_output[0, :, 0].cpu().numpy()  # (P,)
h_ev_np = h_ev_v4[0, :, 0].cpu().numpy()   # (P,)
gate_f0 = gate_np[:, 0]                     # (P,)
ax.plot(dec_np,  color='#1565C0', lw=1.5, label='dec_output (feature 0)')
ax.plot(h_ev_np, color='#D97706', lw=1.5, label='h_ev (= dec × gate)')
ax.set_xlabel('Horizon t')
ax.set_ylabel('Valeur')
ax.set_title('(e) dec_output vs h_ev\n(feature 0, batch 0)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── (f) Comparaison v3 vs v4 : |h_ev - dec_output| ───────────────────────────
ax = fig.add_subplot(gs[1, 2])
diff_v3 = (h_ev_v3 - dec_output).abs().mean(dim=-1)[0].cpu().numpy()  # (P,)
diff_v4 = (h_ev_v4 - dec_output).abs().mean(dim=-1)[0].cpu().numpy()  # (P,)
ax.plot(diff_v3, color='#9D174D', lw=1.5, label=f'v3 (mix=0.25) — fixe')
ax.plot(diff_v4, color='#D97706', lw=1.5, label='v4 (gate × dec) — adaptatif')
ax.set_xlabel('Horizon t')
ax.set_ylabel('|h_ev − dec_output| moyen sur D')
ax.set_title('(f) Déviation de dec_output\nv3 fixe vs v4 adaptatif')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle(
    'Evidence Layer v4 — Sanity Check (tenseurs aléatoires, init std=0.01)\n'
    f'B={B}, P={P}, C={C}, D={D}',
    fontsize=13, fontweight='bold'
)
plt.savefig('evidence_v4_sanity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : evidence_v4_sanity.png')

## 8 — Test gradient (module entraînable)

In [ ]:
# Vérifie que les gradients circulent correctement dans v4
layer_v4_train = EvidenceLayerV4(d_model=D, context_length=C).to(device)
optimizer = torch.optim.Adam(layer_v4_train.parameters(), lr=1e-3)

dec = torch.randn(B, P, D, device=device, requires_grad=True)
enc = torch.randn(B, C, D, device=device, requires_grad=True)

# Cible aléatoire (simule la loss de prédiction)
target = torch.randn(B, P, D, device=device)

gate_dev_before = layer_v4_train.gate_deviation(dec.detach(), enc.detach())

losses = []
for step in range(10):
    optimizer.zero_grad()
    h_ev_t, M_t = layer_v4_train(dec, enc)
    loss = (h_ev_t - target).pow(2).mean()
    loss += evidence_regularization_loss(M_t, alpha=0.0, beta=1e-3, gamma=1e-3)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

gate_dev_after = layer_v4_train.gate_deviation(dec.detach(), enc.detach())

print('Test gradient (10 steps) :')
for i, l in enumerate(losses):
    print(f'  step {i+1:2d}  loss={l:.4f}')
print()
print(f'gate_deviation avant entraînement : {gate_dev_before:.4f}')
print(f'gate_deviation après 10 steps     : {gate_dev_after:.4f}')
print(f'Le gate dévie davantage après entraînement ? {"OUI ✅" if gate_dev_after > gate_dev_before else "NON (normal si loss ↓ trop vite)"}')

# Vérifie que les poids ont bien changé
assert not torch.allclose(
    layer_v4_train.gate_proj.weight,
    torch.zeros_like(layer_v4_train.gate_proj.weight)
), 'gate_proj.weight inchangé → gradient ne circule pas !'
print('Gradients gate_proj : OK ✅')

assert not torch.allclose(
    layer_v4_train.evidence_proj.weight,
    EvidenceLayerV4(D, C).evidence_proj.weight.to(device)
), 'evidence_proj.weight inchangé → gradient ne circule pas !'
print('Gradients evidence_proj : OK ✅')

## 9 — Récapitulatif complet

In [ ]:
print('=' * 68)
print('  RÉCAPITULATIF — Evidence Layer v4 Sanity Check')
print('=' * 68)
print()
print('  Formule (v4) :')
print('    M    = softmax(Linear_D→C(dec_output))          [B,P,C]')
print('    h_ctx = LayerNorm(bmm(M, enc_hidden))            [B,P,D]')
print('    gate  = 1 + tanh(Linear_D→D(h_ctx))             [B,P,D] ∈ (0,2)')
print('    h_ev  = dec_output ⊙ gate                        [B,P,D]')
print()
print('  Comparaison v3 (additif) :')
print('    h_ev  = (1-mix)*dec_output + mix*LayerNorm(bmm(M, enc_hidden))')
print()
print('  Avantages v4 :')
print('    1. gate=1 à l\'init → identité → checkpoint B5 préservé')
print('    2. gate ∈ (0,2) → peut amplifier (>1) ou atténuer (<1)')
print('    3. plus de hyperparamètre mix à ajuster')
print('    4. gate adaptatif par horizon t et par dimension D')
print()
print('  Checks :')

checks = [
    ('h_ev.shape == (B,P,D)', h_ev.shape == (B, P, D)),
    ('M.shape    == (B,P,C)', M.shape    == (B, P, C)),
    ('M ∈ [0,1]',             bool((M >= 0).all() and (M <= 1).all())),
    ('M.sum(dim=-1) ≈ 1',    bool(torch.allclose(M.sum(-1), torch.ones(B, P, device=device), atol=1e-5))),
    ('gate ≈ 1 à l\'init',   diff.max().item() < 1e-2),
    ('Gradients OK',          True),
]

for label, ok in checks:
    status = '✅ OK' if ok else '❌ FAIL'
    print(f'    [{status}] {label}')

all_ok = all(ok for _, ok in checks)
print()
print(f'  Résultat global : {"✅ PASS — prêt pour intégration v4" if all_ok else "❌ FAIL — revoir l\'implémentation"}')
print('=' * 68)

## 10 — Prochaines étapes

Ce notebook valide `EvidenceLayerV4` comme module isolé. Pour la suite :

1. **Intégration dans le modèle principal** : créer `SoftCAMTransformerV4ForPrediction`
   héritant de v3 et remplaçant le mix additif par `EvidenceLayerV4`.

2. **Fine-tuning** : charger le checkpoint B5 (R²=0.7563 avec Fix #5) et
   entraîner quelques époques avec `gate_proj` non gelé.

3. **Comparaison formelle** : reproduire les tests H1.F / H1.G sur v4.
   Hypothèse : H1.F devrait montrer une dégradation plus importante qu'avec
   v3 à mix=0.05, car le gate amplifie/atténue *toute* la représentation
   et non seulement une fraction fixe.

4. **Diagnostic gate_deviation()** :
   - si `gate_deviation ≈ 0` → M uniforme, v4 ≈ identité → pas mieux que v3
   - si `gate_deviation > 0.1` → M concentrée, le gate module effectivement

5. **Figure mémoire** : exporter `evidence_v4_sanity.png` vers
   `redaction/figures/` pour §3.3 du mémoire (Architecture).